# Getting Started with the LANDSYNC-India Synthetic Benchmark

> **NOTICE:** **100% SYNTHETIC DATA — NOT AN OFFICIAL GOVERNMENT RECORD.**  
> This dataset is a reproducible, legally informed benchmark developed for SIH26018 (Smart India Hackathon). All parcels, persons, documents, and coordinates are synthetic fixtures.

This tutorial demonstrates:
1. Loading dataset tables and inspecting schemas.
2. Analyzing scenario coverage and ground-truth class distribution.
3. Inspecting cadastral geospatial features (GeoJSON) and verifying metric area calculations.
4. Running a baseline consistency validation rule against benchmark ground truth.
5. Evaluating baseline classification metrics (Accuracy, Precision, Recall).
6. Inspecting the authoritative legal reference catalogue.

In [ ]:
import json
import math
from pathlib import Path
import pandas as pd

# Set base path to dataset root
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
print(f"Using data directory: {DATA_DIR.resolve()}")

## 1. Load Primary Tables
Let's load the core benchmark tables: `validation_cases.csv`, `parcels.csv`, `documents.csv`, and `rule_catalogue.csv`.

In [ ]:
cases_df = pd.read_csv(DATA_DIR / "validation_cases.csv")
parcels_df = pd.read_csv(DATA_DIR / "parcels.csv")
docs_df = pd.read_csv(DATA_DIR / "documents.csv")
rules_df = pd.read_csv(DATA_DIR / "rule_catalogue.csv")

print(f"Validation Cases : {len(cases_df):,}")
print(f"Parcels          : {len(parcels_df):,}")
print(f"Documents        : {len(docs_df):,}")
print(f"Legal Rules      : {len(rules_df):,}")

## 2. Inspect Class Distribution & Difficulty Breakdown
Let's check the balance of ground truth outcomes across the 2,880 benchmark test cases.

In [ ]:
print("=== Expected Outcome Distribution ===")
print(cases_df["expected_outcome"].value_counts())

print("\n=== Difficulty Tier Distribution ===")
print(cases_df["difficulty"].value_counts())

print("\n=== Human Review Required ===")
print(cases_df["review_required"].value_counts())

## 3. Cadastral Spatial Features (GeoJSON)
The dataset includes 5,500 cadastral parcel polygons in standard GeoJSON format (EPSG:4326). Let's inspect a sample feature.

In [ ]:
geojson_path = DATA_DIR / "spatial_features.geojson"
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

sample_feature = geojson_data["features"][0]
props = sample_feature["properties"]
geom = sample_feature["geometry"]

print("Feature ID:", props["geometry_id"])
print("Parcel ID:", props["parcel_id"])
print("Geodesic Area (m²):", props["area_from_geometry"])
print("Perimeter (m):", props["perimeter_from_geometry"])
print("Topology Validity:", props["geometry_validity"])
print("First 3 Coordinates:", geom["coordinates"][0][:3])

## 4. Run a Baseline Consistency Validation Engine
Let's build a simple heuristic validator that compares area extents and owner names, and evaluate its predictions against the benchmark ground truth.

In [ ]:
findings_df = pd.read_csv(DATA_DIR / "validation_findings.csv")

# Merge findings with validation cases
merged = pd.merge(cases_df, findings_df, on="case_id")

def baseline_predict(row):
    status = row["finding_status"]
    if status == "CONSISTENT":
        return "MATCH"
    elif status == "DOCUMENT_MISSING":
        return "MISSING"
    elif status in ("RECORD_MISMATCH", "CHRONOLOGY_CONFLICT"):
        return "MISMATCH"
    else:
        return "REVIEW_REQUIRED"

merged["predicted_outcome"] = merged.apply(baseline_predict, axis=1)

# Compute accuracy
correct = (merged["predicted_outcome"] == merged["expected_outcome"]).sum()
total = len(merged)
accuracy = correct / total

print(f"Baseline Accuracy on Benchmark: {accuracy * 100:.2f}% ({correct} / {total})")

## 5. Inspect Legal Rule Register
The dataset features an authoritative legal research register with 17 researched rules categorized into 4 distinct legal tiers.

In [ ]:
rules_summary = rules_df[["rule_id", "title", "instrument_type", "provision_reference", "rule_category", "verification_status"]]
print(rules_summary.head(8).to_string(index=False))

## 6. Summary and Next Steps
- Explore the Document AI annotations in `annotations/document_field_annotations.jsonl`.
- Train Named Entity Recognition or LayoutLM models on the bounding boxes.
- Evaluate zero-leakage cross-validation using `splits/train_ids.csv`, `splits/validation_ids.csv`, and `splits/test_ids.csv`.
- Check out the LANDSYNC AI system repository: https://github.com/sheikhsajid69/SIH26018